In [ ]:
using Pkg
Pkg.activate(".")
Pkg.instantiate()

In [ ]:
using Plots, BenchmarkProfiles, DataFrames, Printf, ForwardDiff, CSV

In [ ]:
const RESULTS_DIR = "results_timed"

In [ ]:
global fig_counter = 1

function save_profile(plt, header, metric)
    global fig_counter
    savefig(plt, "tmp/"*"fig$(fig_counter)_"*header*"_"*String(metric)*".pdf")
    fig_counter += 1
    return 
end


# Comparing TRAULLS with different Hessian approximations

In [ ]:
# Store results dataframes
traulls_variants = Dict{String, DataFrame}()
traulls_variants["Gauss-Newton"] = CSV.read(RESULTS_DIR*"/traulls_gn.csv", DataFrame)
traulls_variants["BFGS"] = CSV.read(RESULTS_DIR*"/traulls_bfgs.csv", DataFrame)
traulls_variants["SR1"] = CSV.read(RESULTS_DIR*"/traulls_sr1.csv", DataFrame)
traulls_variants["Hybrid-BFGS"] = CSV.read(RESULTS_DIR*"/traulls_hybrid_bfgs.csv", DataFrame)
traulls_variants["Hybrid-SR1"] = CSV.read(RESULTS_DIR*"/traulls_hybrid_sr1.csv", DataFrame)

df_traulls = values(traulls_variants);

In [ ]:
# Find instances with different solutions

pb_names = first(df_traulls)[!,:name]
to_remove = []

for i in axes(pb_names, 1)
    if all(df -> df[i, :status] == "first_order_critical", df_traulls)
        obj_values = (df -> df[i,:objective]).(df_traulls)
        fmax, fmin = maximum(obj_values), minimum(obj_values)
        if (fmax-fmin) / (1 + max(abs(fmin), abs(fmax))) > 1e-2
            push!(to_remove, pb_names[i])
        end
    end
end

# Remove instances
for v in values(traulls_variants)
    filter!(row -> !(row.name in to_remove), v)
end

to_remove

In [ ]:
function make_traulls_profile(metric, solvers, dict_lstyle; plot_title="")

    hess_variants = collect(keys(solvers))
    T = zeros(nrow(first(solvers).second), length(solvers))
    
    for (j, df) in enumerate(values(solvers))
        for (i, row) in enumerate(eachrow(df))
            T[i,j] = row[:status] == "first_order_critical" ? row[metric] : Inf
        end
    end
    
    performance_profile(PlotsBackend(), T, collect(keys(solvers)), 
        palette=:tab10, linestyles = [dict_lstyle[s] for s in hess_variants],
        legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = plot_title)
end

## All Hessians

In [ ]:
lstyles = Dict("Gauss-Newton" => :solid, 
    "BFGS" => :solid, 
    "SR1" => :solid,
    "Hybrid-BFGS" => :dash, 
    "Hybrid-SR1" => :dot)

metrics = Dict(:elapsed_time => "elapsed time", :nouter_iter => "outer iterations", :ninner_iter => "inner iterations")

for (k, v) in metrics
    plt = make_traulls_profile(k, traulls_variants, lstyles; plot_title = v)
    # save_profile(plt, "traulls_variants", k)
    display(plt)
end

## Comparing Gauss-Newton and SR1 (plain and hybrid variants) on problems with similar residuals magnitude

In [ ]:
function stratified_profile(metric, solvers, strat, dict_lstyle; plot_title="")

    hess_variants = collect(keys(solvers))
    T = zeros(length(strat), length(solvers))
    
    for (j, df) in enumerate(values(solvers))
        for (i, row) in enumerate(eachrow(filter(row -> row.name in strat, df)))
            T[i,j] = row[:status] == "first_order_critical" ? row[metric] : Inf
        end
    end
    
    performance_profile(PlotsBackend(), T, collect(keys(solvers)), 
        palette=:tab10, linestyles = [dict_lstyle[s] for s in hess_variants],
        legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = plot_title, 
        titlefontsize=8, guidefontsize=8)
end

In [ ]:
df_gn = traulls_variants["Gauss-Newton"]
zero_res = filter(row -> 2*row.objective < 1e-8, df_gn)[!,:name]
small_res = filter(row -> 1e-8 <= 2*row.objective < 1.0, df_gn)[!,:name]
medium_res = filter(row -> 1.0 <= 2*row.objective < 1e2, df_gn)[!,:name]
large_res = filter(row -> 1e2 <= 2*row.objective, df_gn)[!,:name]

variants_for_stratified = Dict(k => traulls_variants[k] for k in ["Gauss-Newton", "SR1", "Hybrid-SR1"])
lstyles = Dict("Gauss-Newton" => :solid, "SR1" => :dash, "Hybrid-SR1" => :dot)
metrics = Dict(:elapsed_time => "elapsed time", :neval_residual => "residuals evaluations")

for (k, v) in metrics
    plt_zero = stratified_profile(k, variants_for_stratified, zero_res, lstyles; 
        plot_title = v * " - $(length(zero_res)) problems with ||rₒₚₜ||² ≤ 10⁻⁸")

    # save_profile(plt_zero, "zero_res", k)
    plt_small = stratified_profile(k, variants_for_stratified, small_res, lstyles; 
        plot_title = v * " - $(length(small_res)) problems with 10⁻⁸ ≤ ||rₒₚₜ||² < 1")
    # save_profile(plt_small, "small_res", k)
    plt_medium = stratified_profile(k, variants_for_stratified, medium_res, lstyles; 
        plot_title = v * " - $(length(medium_res)) problems with 1 ≤ ||rₒₚₜ||² < 100")
    # save_profile(plt_medium, "medium_res", k)
    plt_large = stratified_profile(k, variants_for_stratified, large_res, lstyles; 
        plot_title = v * " - $(length(large_res)) problems with ||rₒₚₜ||² ≥ 100")
    # save_profile(plt_large, "large_res", k)

    plt = plot(plt_zero, plt_small, plt_medium, plt_large, layout = 4)
    display(plt)
end

# Comparison against IPOPT and Percival

In [ ]:
function solvers_comparison_profile(metric, solvers, stats_traulls, stats_ipopt, stats_percival; plot_title="")
    T = zeros(nrow(stats_traulls), 3)

    for (i, row) in enumerate(eachrow(stats_traulls))
        T[i, 1] = row[:status] == "first_order_critical" ? row[metric] : Inf
    end

    for (i, row) in enumerate(eachrow(stats_ipopt))
        T[i, 2] = row[:status] == "first_order" ? row[metric] : Inf
    end

    for (i, row) in enumerate(eachrow(stats_percival))
        T[i, 3] = row[:status] == "first_order" ? row[metric] : Inf
    end
    
    performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:dot, :solid, :dash],
        legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = plot_title)
end

In [ ]:
# Make performance profile for elapsed time, residuals and gradient evaluations

solvers = ["TRAULLS", "IPOPT", "Percival"]
metrics = Dict(:elapsed_time => "elapsed time", :neval_residual => "residuals evaluations", :neval_grad => "gradient evaluations")

for (k, v) in metrics
    plt = solvers_comparison_profile(k, solvers, stats_traulls, stats_ipopt, stats_percival; plot_title=v)
    display(plt)
end

## IPOPT with L-BFGS

In [ ]:
# Reload Traulls (Hybrid SR1 variant) results
df_traulls = CSV.read(RESULTS_DIR*"/traulls_hybrid_sr1.csv", DataFrame)

# Load results for IPOPT (limited memory hessian) and Percival results
df_ipopt_lbfgs = CSV.read(RESULTS_DIR*"/ipopt-lbfgs.csv", DataFrame)
df_percival = CSV.read(RESULTS_DIR*"/percival.csv", DataFrame);

In [ ]:
function solvers_comparison_profile(metric, solvers, stats_traulls, stats_ipopt, stats_percival; plot_title="")
    T = zeros(nrow(stats_traulls), 3)

    for (i, row) in enumerate(eachrow(stats_traulls))
        T[i, 1] = row[:status] == "first_order_critical" ? row[metric] : Inf
    end

    for (i, row) in enumerate(eachrow(stats_ipopt))
        T[i, 2] = row[:status] == "first_order" ? row[metric] : Inf
    end

    for (i, row) in enumerate(eachrow(stats_percival))
        T[i, 3] = row[:status] == "first_order" ? row[metric] : Inf
    end
    
    performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:dot, :solid, :dash],
        legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = plot_title)
end

In [ ]:
# Remove instances with different objective functions

pb_names = df_traulls[!,:name]
to_remove = []
for i in 1:nrow(df_traulls)
    if df_traulls[i,:status] == "first_order_critical" && df_ipopt_lbfgs[i,:status] == "first_order" && df_percival[i, :status] =="first_order"
        obj_values = (df -> df[i, :objective]).([df_traulls, df_ipopt_lbfgs, df_percival])
        fmax, fmin = maximum(obj_values), minimum(obj_values)
        if (fmax-fmin) / (1 + max(abs(fmin), abs(fmax))) > 1e-2
            println(pb_names[i])
            push!(to_remove, pb_names[i])
        end
    end
end

stats_traulls, stats_ipopt_lbfgs, stats_percival = (filter(row -> !(row.name in to_remove))).([df_traulls, df_ipopt_lbfgs, df_percival]);

In [ ]:
solvers = ["TRAULLS", "IPOPT-LBFGS", "Percival"]
metrics = Dict(:elapsed_time => "elapsed time", :neval_residual => "residuals evaluations", :neval_grad => "gradient evaluations")

for (k, v) in metrics
    plt = solvers_comparison_profile(k, solvers, stats_traulls, stats_ipopt_lbfgs, stats_percival; plot_title=v)
    # save_profile(plt, "traulls_vs_others", k)
    display(plt)
end